# 08. Zero-Copy Views & Stride Tricks: Beginner Guide

### 📌 Overview & Architectural Context
Welcome to **08. Zero-Copy Views & Stride Tricks**. NumPy's stride mechanics allow creating virtual multi-dimensional views over existing memory buffers with zero RAM copying. This notebook explores sliding window transformations, 1D rolling time-series features, 2D convolution patch extraction, and the modern safe API `np.lib.stride_tricks.sliding_window_view()` alongside low-level `as_strided()`.

### 📚 Key Concepts Covered in this Notebook:
- [x] 🔹 Low-Level Windowing with `as_strided()`
- [x] 🔹 Zero-RAM Moving Averages via Stride Tricks
- [x] 🔹 Safe Sliding Windows with `sliding_window_view()`
- [x] 🔹 D Patch Extraction on Matrix


In [1]:
# Setup imports & dataset loading from raw_transactions.csv
import numpy as np
import pandas as pd
import sys
import time
import os

# Load raw transactions and extract aligned NumPy numeric arrays
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
raw_df = pd.read_csv(csv_path)
clean_raw = raw_df.dropna(subset=['transaction_amount', 'is_fraud', 'account_age_months']).reset_index(drop=True)
amounts = clean_raw['transaction_amount'].to_numpy(dtype=np.float64)
fraud_flags = clean_raw['is_fraud'].to_numpy(dtype=np.int8)
account_ages = clean_raw['account_age_months'].to_numpy(dtype=np.float32)

print(f"NumPy Version: {np.__version__}")
print(f"Loaded from {csv_path} ({len(amounts)} clean aligned rows):")
print(f"- amounts array: shape {amounts.shape}, dtype {amounts.dtype}")
print(f"- fraud_flags array: shape {fraud_flags.shape}, dtype {fraud_flags.dtype}")
print(f"- account_ages array: shape {account_ages.shape}, dtype {account_ages.dtype}")

NumPy Version: 1.26.4
Loaded from ../data/raw_transactions.csv (14251 clean aligned rows):
- amounts array: shape (14251,), dtype float64
- fraud_flags array: shape (14251,), dtype int8
- account_ages array: shape (14251,), dtype float32


### 🔹 Low-Level Windowing with `as_strided()`
- **What it does:** Constructs a 5-elements zero-copy sliding window matrix over the amounts vector with zero RAM overhead.
- **Syntax:** `as_strided()`
- **Key Note:** Window calculations must be chained with an aggregation function (like `.mean()`, `.sum()`, or `.std()`) to compute the final windowed summary.
- **Dataset Application & Code Demonstration:** Applies Low-Level Windowing with `as_strided()` across the extracted numeric transaction `amounts` array to compute performance metrics.


In [2]:
from numpy.lib.stride_tricks import as_strided
win_sz = 5
n_win = len(amounts[:20]) - win_sz + 1
stride_b = amounts.itemsize
rolling_amounts = as_strided(amounts[:20], shape=(n_win, win_sz), strides=(stride_b, stride_b))
print('Zero-Copy Rolling Window Matrix (first 3 windows):\n', rolling_amounts[:3].round(2))
print('Shares memory buffer with original?:', rolling_amounts.base is not None)

Zero-Copy Rolling Window Matrix (first 3 windows):
 [[ 607.78 1819.11   64.08 1025.73  772.74]
 [1819.11   64.08 1025.73  772.74  198.47]
 [  64.08 1025.73  772.74  198.47  217.23]]
Shares memory buffer with original?: True


### 🔹 Zero-RAM Moving Averages via Stride Tricks
- **What it does:** Calculates moving average numerical values instantaneously as `rolling_amounts.mean(axis=1)`.
- **Syntax:** `function(*args, **kwargs)`
- **Key Note:** NumPy operations are optimized for homogeneous numeric data, offering massive speed improvements over standard Python loops.
- **Dataset Application & Code Demonstration:** Applies Zero-RAM Moving Averages via Stride Tricks across the extracted numeric transaction `amounts` array to compute performance metrics.


In [3]:
moving_avg_spend = rolling_amounts.mean(axis=1)
print('Moving Average Spend (window=5):', moving_avg_spend[:5].round(2))

Moving Average Spend (window=5): [857.89 776.03 455.65 656.97 715.95]


### 🔹 Safe Sliding Windows with `sliding_window_view()`
- **What it does:** NumPy 1.20+ safe stride wrapper avoiding segmentation faults.
- **Syntax:** `sliding_window_view()`
- **Key Note:** Window calculations must be chained with an aggregation function (like `.mean()`, `.sum()`, or `.std()`) to compute the final windowed summary.
- **Dataset Application & Code Demonstration:** Applies Safe Sliding Windows with `sliding_window_view()` across the extracted numeric transaction `amounts` array to compute performance metrics.


In [4]:
from numpy.lib.stride_tricks import sliding_window_view
safe_rolling = sliding_window_view(amounts[:20], window_shape=5)
print('Safe Sliding Window Matrix (first 3):\n', safe_rolling[:3].round(2))

Safe Sliding Window Matrix (first 3):
 [[ 607.78 1819.11   64.08 1025.73  772.74]
 [1819.11   64.08 1025.73  772.74  198.47]
 [  64.08 1025.73  772.74  198.47  217.23]]


### 🔹 D Patch Extraction on Matrix
- **What it does:** Extracts 3x3 patches from a 2D elements correlation grid with zero memory duplication.
- **Syntax:** `function(*args, **kwargs)`
  - **Parameters:**
    - `row_label` (*hashable*): Row label.
    - `col_label` (*hashable*): Column label.
- **Key Note:** NumPy operations are optimized for homogeneous numeric data, offering massive speed improvements over standard Python loops.
- **Dataset Application & Code Demonstration:** Applies D Patch Extraction on Matrix across the extracted numeric transaction `amounts` array to compute performance metrics.


In [5]:
mock_grid = amounts[:25].reshape(5, 5)
patches = sliding_window_view(mock_grid, window_shape=(3, 3))
print('Grid Shape:', mock_grid.shape, 'Patches 4D Shape:', patches.shape)
print('First 3x3 Patch:\n', patches[0, 0].round(2))

Grid Shape: (5, 5) Patches 4D Shape: (3, 3, 3, 3)
First 3x3 Patch:
 [[ 607.78 1819.11   64.08]
 [ 198.47  217.23 1070.66]
 [1347.73  529.53 1668.79]]


## 💡 Real-World Practice & Scenarios
Practical scenarios and common data engineering questions explained with real examples.


### 🔍 Scenario: Q1: Zero-RAM Rolling Volatility in Transaction Flow
- **Objective:** Q1: Zero-RAM Rolling Volatility in Transaction Flow
- **Approach:** Compute 10-period rolling standard deviation over transaction amounts using zero-copy sliding windows.
- **Syntax:** `sliding_window_view(amounts, 10).std(axis=-1)`

In [6]:
rolling_vol = sliding_window_view(amounts[:100], 10).std(axis=-1)
print('10-Period Rolling Spending Volatility Head:', rolling_vol[:5].round(2))

10-Period Rolling Spending Volatility Head: [524.5  548.32 445.34 470.94 492.09]
